# RocmPilot — AMD (MI300X/Radeon) validation capture
Run the cells top-to-bottom in an AMD ROCm PyTorch environment. It writes
`validation_log.txt` + `benchmark.json` in RocmPilot's exact fixture format.
Paste both back (or download them) so replay mode shows genuine AMD numbers.

In [ ]:
# 1) Sanity check — confirm we're on the GPU and read the REAL device name.
import torch
print('torch    :', torch.__version__)
print('hip      :', getattr(torch.version, 'hip', None))
print('cuda ok  :', torch.cuda.is_available())
print('gpu name :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) Capture: smoke test + benchmark in RocmPilot's exact format.
import json, subprocess, time, torch

SIZE, N_RUNS = 1024, 50
_CPU_HINTS = ('EPYC', 'Ryzen', 'Threadripper', 'Core Processor', 'Xeon', 'Intel')

def gpu_name():
    # torch is the source of truth; rocminfo fallback must skip the CPU agent.
    if torch.cuda.is_available():
        try: return torch.cuda.get_device_name(0)
        except Exception: pass
    try:
        out = subprocess.run(['rocminfo'], capture_output=True, text=True, timeout=30).stdout
        names = [l.split(':',1)[1].strip() for l in out.splitlines() if 'Marketing Name' in l]
        gpus = [n for n in names if not any(h in n for h in _CPU_HINTS)]
        return gpus[0] if gpus else (names[0] if names else 'cpu')
    except Exception:
        return 'cpu'

lines = []
def out(s=''): print(s); lines.append(s)

accel = torch.cuda.is_available()
hip = getattr(torch.version, 'hip', None)
name = gpu_name()
dev = torch.device('cuda' if accel else 'cpu')

out('=== RocmPilot AMD Validation (LIVE — AMD Developer Cloud) ===')
out(f'Runtime    : ROCm {hip or "(not a ROCm build)"}')
out('')
out('$ python smoke_test.py --require-gpu')
out('=== RocmPilot smoke test ===')
out(f'torch version        : {torch.__version__}')
out(f'HIP / ROCm version   : {hip or "not a ROCm build"}')
out(f'accelerator available: {accel}')
out(f'device name          : {name}')
out(f'selected device      : {dev}')
a = torch.randn(512, 512, device=dev); b = torch.randn(512, 512, device=dev)
c = (a @ b).sum().item()
if dev.type == 'cuda': torch.cuda.synchronize()
ok = bool(torch.isfinite(torch.tensor(c)).item())
out(f'matmul result finite : {ok}')
out('PASS' if ok else 'FAIL')

out(''); out('$ python benchmark.py')
t0 = time.perf_counter()
a = torch.randn(SIZE, SIZE, device=dev); b = torch.randn(SIZE, SIZE, device=dev)
load_ms = (time.perf_counter() - t0) * 1000
for _ in range(5): _ = a @ b
if dev.type == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N_RUNS): _ = a @ b
if dev.type == 'cuda': torch.cuda.synchronize()
avg_ms = (time.perf_counter() - t0) * 1000 / N_RUNS

res = {'device': str(dev), 'gpu_name': name, 'pytorch_build': torch.__version__,
       'hip_version': hip, 'load_ms': round(load_ms, 3),
       'inference_latency_ms': round(avg_ms, 3),
       'approx_tflops': round((2 * SIZE**3) / (avg_ms / 1000) / 1e12, 2),
       'runs': N_RUNS, 'matmul_size': SIZE}
out(json.dumps(res, indent=2))
out('')
out('RESULT: AMD inference smoke test ' + ('PASSED. Benchmark captured.' if ok else 'FAILED.'))

open('validation_log.txt', 'w').write('\n'.join(lines) + '\n')
open('benchmark.json', 'w').write(json.dumps(res, indent=2))
print('\n>>> wrote validation_log.txt and benchmark.json')

In [ ]:
# 3) Show both files so you can copy them back.
print(open('validation_log.txt').read())
print('----- benchmark.json -----')
print(open('benchmark.json').read())